# Creating Tax Assessments with Feature Engineering
## Practice Skeleton

**Short name (GitHub):** `TaxCreate`

**Adaptation of:** `LaptopPrice` (unit-bearing strings → numeric features → encode → Random Forest).

**Card:** `data/tax_returns.csv` — **920** draft returns × **23** columns. Target `tax_due` is the assessment the desk *creates* (USD; mean ≈ $6,299, median $4,699, about 16% zeros). Money, percents, dependents, pages, stars, and tax year arrive as text (`"$62,000"`, `"5.50%"`, `"2 dependents"`, `"8-page"`, `"4 stars"`, `"2023 TY"`).

**How to use**
- Fill cells marked `# YOUR CODE HERE`. Keep the cheat-sheet and flowchart visible.
- Compare with `TaxCreate_Solution.ipynb` only after an attempt.
- Data: `data/tax_returns.csv`. Charts: `taxcreate_*.png`.
- Clone the pipeline with `TaxCreate_Reusable_Template.ipynb`.
- **Not a filing engine, not tax advice, not an audit determination.** Teaching assessments only.


## Inline cheat-sheet (keep this cell visible)

See also **`TaxCreate_Cheatsheet.docx`**.

| Item | Code / rule |
|------|-------------|
| Money strings | `.astype(str).str.replace('$','').str.replace(',','').astype(float)` |
| Percent | strip `'%'` then `/ 100` |
| Composite | `gross_income = wages + business + capgains` (the AGI-like total) |
| Dependents | `.str.extract(r'(\d+)')` |
| Pages | strip `'-page'` |
| Sentinel year | `'Not Available'` → `'0'` *before* stripping `' TY'` |
| Cardinality | `< 5` uniques → one-hot `drop_first`. `≥ 5` → target-mean `tax_due` |
| Leakage | Fit means on **train only**. Unseen test labels → `y_train.mean()` |
| Split | `test_size=0.2`, `random_state=42` |
| Model | `RandomForestRegressor(random_state=42)` |
| Metrics | MAE in dollars + R². Always quote the **mean-predictor baseline**. |
| Never | Ship this as a substitute for the statutory calculator. |


## Flowchart of the desired outcome

![TaxCreate flow](taxcreate_flowchart.png)

Load the 920-row extract → strip `$` `,` `%` from money and rate columns → engineer `gross_income` → clean stars, dependents, pages, tax year → EDA → encode by cardinality → 80/20 split → Random Forest vs linear / mean baselines → MAE + R² + top features → poke `n_estimators`, depth, noise, and sample size.


## 0. Packages


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid")


## 1. Load and inspect the return extract

920 rows. Columns that *look* numeric (`wages_usd`, `state_rate`, `dependents`, `compliance_stars`) are `object` because of `$`, `%`, and words.

**Task**
- Load `data/tax_returns.csv` into `df`.
- Print `.head()`, `.info()`, `.shape`, `.nunique()`, and `df['tax_due'].describe()`.


In [ ]:
# YOUR CODE HERE
df = None


## 2. Parse money columns and engineer gross income

Loop over `['wages_usd', 'business_usd', 'capgains_usd', 'deduction_usd', 'credits_usd']`.
Strip `$` and `,`, cast to float.

Create `gross_income = wages_usd + business_usd + capgains_usd`.

Print the five money columns plus `gross_income` for the first rows.


In [ ]:
# YOUR CODE HERE


## 3. Clean rate, stars, year, dependents, pages

Idempotent pattern: `.astype(str)` first.

- `state_rate`: strip `%`, cast float, divide by 100.
- `compliance_stars`: strip `" stars"` then `" star"` → int.
- `tax_year`: `"Not Available"` → `"0"`, strip `" TY"` → int. Store as `tax_year_n` and drop `tax_year`.
- `dependents`: extract the leading digits → int.
- `return_pages`: strip `"-page"` → int.


In [ ]:
# YOUR CODE HERE


## 4. Exploratory data analysis

- Correlation heatmap on numeric columns.
- Boxplot `dependents` vs `tax_due`.
- Boxplot `state_rate` vs `tax_due`.
- Histogram of `tax_due` (right skew + a zero mass).

![heatmap](taxcreate_heatmap.png)
![dependents](taxcreate_deps_box.png)
![rate](taxcreate_rate_box.png)


In [ ]:
# YOUR CODE HERE
numeric_df = None


## 5. Encode categoricals and split

Object columns remaining should be things like `tax_office`, `income_class`, `filing_status`, `filer_channel`, `deduction_type`, `complexity`, `filing_timing`, `digital_review`, `paid_preparer`, `state_resident`.

- `nunique() < 5` → one-hot, `drop_first=True`.
- else → map to mean `tax_due` (lesson version uses the full frame — note the leakage).

Then 80/20 split, `random_state=42`.

**Stretch:** split first; fit target means on train only.


In [ ]:
# YOUR CODE HERE
X_train = X_test = y_train = y_test = None


## 6. Fit the Random Forest

`RandomForestRegressor(random_state=42)` → `y_pred`.


In [ ]:
# YOUR CODE HERE
rf_model = None
y_pred = None


## 7. Evaluate and rank features

Print MAE, RMSE, R², and the mean-predictor baseline MAE.

Top-5 impurity importances + horizontal bar.

![importance](taxcreate_importance.png)
![pred vs actual](taxcreate_pred_actual.png)


In [ ]:
# YOUR CODE HERE
mae = r2 = None


## 8. Alternate code — same destination, different route


### 8a. Regex extract for money and dependents


In [ ]:
# YOUR CODE HERE
# df['wages_usd'].astype(str).str.replace('[^0-9.]', '', regex=True)


### 8b. Leakage-safe target encoding


In [ ]:
# YOUR CODE HERE


### 8c. Linear / Ridge on the same matrix


In [ ]:
# YOUR CODE HERE


### 8d. Permutation importance + a statutory rebuild


In [ ]:
# YOUR CODE HERE
# permutation_importance(...)
# Optional: taxable ≈ gross - deduction - 1800*dependents; compare MAE of a hand formula.


## 9. More practice

1. Predict `np.log1p(tax_due)` then `np.expm1` the predictions.
2. Drop `Number of Notices` and `Number of Workpapers` (desk activity, not a filing input). How much R² do you lose?
3. Fit only `income_class == 'Wages'`.
4. Flag “balance due” as `tax_due >= 5000` and report tail MAE.
5. Rebuild a mini statutory engine (`taxable = max(0, gross - deduction - 1800*deps)`) and compare its MAE to the forest.


In [ ]:
# YOUR CODE HERE — pick at least two


## 10. Simulation

Reference: ![simulation](taxcreate_simulation.png)

Edit knobs. Default reference: MAE ≈ **$1,149**, R² ≈ **0.923**.


In [ ]:
N_EST = 100
MAX_DEPTH = None
NOISE_SD = 0          # dollars added to y_train (try 400, 1500)
SUBSAMPLE = 1.0
RANDOM_STATE = 42


In [ ]:
# YOUR CODE HERE


## 11. What this model can and cannot do

**Can**
- Reconstruct most of a noisy statutory assessment from parsed money fields (gross income dominates).
- Beat a mean assessment guess by a wide margin (~$1.15k vs ~$4.85k MAE).
- Show that credits and deductions move the bill the way a desk would expect.

**Cannot**
- Replace the published rate tables or a tax-prep engine.
- Price a filing in another country or another year without a refresh.
- Use notice counts as a *cause* of tax_due — they arrive after the assessment exists.
- Decide who gets audited.

**Same pipeline:** payroll withholding estimates, VAT invoices, property-tax bills, customs duties, corporate estimated payments.
